## Preprocessing
Here are the functions needed to build a normalized RAG schema.
### Steps:
- Parse the blocks in a recipe file.
- Extracting metadata (tags, info like prep time, ingredients)
- Put the block into a jsonl schema.

In [8]:
from pathlib import Path
import re
from typing import Dict, Optional, List, Any, Tuple, Sequence
import hashlib
from datetime import datetime
import json

In [9]:
def stable_id(text: str, prefix: str = "doc") -> str:
    h = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:16]
    return f"{prefix}_{h}"

def normalize_whitespace(raw_str: str) -> str:
    """
    Normalize the whitespaces over a string.
    """
    raw_str = raw_str.replace("\r\n", "\n").replace("\r", "\n")
    raw_str = re.sub(r"[ \t]+", " ", raw_str)
    raw_str = re.sub(r"\n{3,}", "\n\n", raw_str)
    return raw_str.strip()

def split_sections(raw: str, sections: List[str]) -> Dict[str, str]:
    """
    Split name, ingredients, info, instructions sections.
    """
    raw = normalize_whitespace(raw)
    
    pattern = re.compile(           # section header pattern
        r"^(name|ingredients|info|instructions)\s*:\s*$", re.IGNORECASE | re.MULTILINE
        )

    matches = list(pattern.finditer(raw))
    if not matches:
        # If no headers found, treat entire thing as instructions (fallback)
        return {"name": "", "ingredients": "", "info": "", "instructions": raw}

    sections: Dict[str, str] = {key: "" for key in sections}

    for idx, match in enumerate(matches):
        key = match.group(1).lower()
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(raw)
        content = raw[start:end].strip()
        sections[key] = content

    return sections

def parse_minutes(value: str) -> Optional[int]:
    """
    Convert time indications to minutes, returns 'None' if string is not parsable
    """
    value = value.strip().lower()

    # Common patterns
    # "60 min" / "60 mins"
    mins = re.search(r"(\d+)\s*(min|mins|minute|minutes)\b", value)
    # "1 h 15 min"
    hrs = re.search(r"(\d+)\s*(h|hr|hrs|hour|hours)\b", value)

    if not mins and not hrs:
        return None

    minutes = 0
    if hrs:
        minutes += int(hrs.group(1)) * 60
    if mins:
        minutes += int(mins.group(1))

    return minutes if minutes > 0 else None

def parse_servings(value: str) -> Optional[int]:
    """
    Parse the no. of servings stated in the recipe.
    """
    v = value.strip()
    m = re.search(r"(\d+)", v)
    return int(m.group(1)) if m else None

def parse_info_block(info_text: str) -> Dict[str, Any]:
    """
    Parses lines like:
      Difficulty: Average
      Prep time: 60 min
      Cook time: 45 min
      Serving: 8
      Cost: Average
      Note + chilling time...
    """
    out: Dict[str, Any] = {
        "difficulty": None,
        "cost": None,
        "prep_time": None,
        "cook_time": None,
        "total_time": None,
        "servings": None,
        "notes": []
    }

    lines = [ln.strip() for ln in info_text.split("\n") if ln.strip()]
    for ln in lines:
        # Key: Value
        kv = re.match(r"^([A-Za-z \-\+]+)\s*:\s*(.+)$", ln)
        if kv:
            key = kv.group(1).strip().lower()
            val = kv.group(2).strip()

            if key in {"difficulty"}:
                out["difficulty"] = val
            elif key in {"cost"}:
                out["cost"] = val
            elif key in {"prep time", "prep_time"}:
                out["prep_time"] = parse_minutes(val)
            elif key in {"cook time", "cook_time"}:
                out["cook_time"] = parse_minutes(val)
            elif key in {"serving", "servings", "yield"}:
                out["servings"] = parse_servings(val)
            else:
                # Preserve unknown structured lines as notes
                out["notes"].append(ln)
        else:
            # Free-form note line
            out["notes"].append(ln)

    # Compute total_time if possible
    if out["prep_time"] is not None or out["cook_time"] is not None:
        out["total_time"] = (out["prep_time"] or 0) + (out["cook_time"] or 0)

    return out

def parse_ingredients_block(ingredients_text: str) -> List[Dict[str, str]]:
    """
    Parses ingredient lines like:
    Type 00 flour: 3cups(380 g)
    Nutmeg: to taste
    """
    NAME_PART  = r"^\s*(.+?)\s*"
    SEP_PART   = r"\s*:\s*"
    VALUE_PART = r"(.+?)\s*$"
    INGREDIENT_LINE_RE = re.compile(f"{NAME_PART}{SEP_PART}{VALUE_PART}")
    items: List[Dict[str, str]] = []
    for line in ingredients_text.split("\n"):

        line = line.strip()
        if not line:                # whitespace
            continue
        match = INGREDIENT_LINE_RE.match(line)
        if match:
            name = match.group(1).strip()
            amount = match.group(2).strip()
        else:                       # no match -> consider line as name
            name, amount = line, ""
        items.append({"name": name, "amount": amount})
    return items

def split_instructions(instructions_text: str) -> List[str]:
    """
    Split line into steps. Split newline when present, or split by sentence boundary.
    """
    text = normalize_whitespace(instructions_text)

    # Separate newlines if present.
    if "\n" in instructions_text:
        parts = [normalize_whitespace(p) for p in instructions_text.split("\n") if p.strip()]
        # Merge very short fragments into previous step
        steps: List[str] = []
        for p in parts:
            if steps and len(p) < 40:
                steps[-1] = (steps[-1] + " " + p).strip()
            else:
                steps.append(p)
        return steps

    # Or do a sentence split if newlines are not present.
    sents = re.split(r"(?<=[.!?])\s+", text)
    sents = [s.strip() for s in sents if s.strip()]

    # Group into 2-3 sentences per step
    steps: List[str] = []
    buf: List[str] = []
    for s in sents:
        buf.append(s)
        if len(buf) >= 3:
            steps.append(" ".join(buf))
            buf = []
    if buf:
        steps.append(" ".join(buf))
    return steps

def infer_tags(dish_name: str, ingredients: List[Dict[str, str]], instructions: str) -> List[str]:
    """
    Simple tag inferer.
    """
    name_l = (dish_name or "").lower()
    ing_l = " ".join([i["name"].lower() for i in ingredients])
    txt = (name_l + " " + ing_l + " " + instructions.lower())

    tags = set()

    if any(k in txt for k in ["tart", "cake", "cookie", "dessert", "chocolate", "sugar", "icing", "ganache"]):
        tags.add("dessert")
    if any(k in txt for k in ["pasta", "spaghetti", "penne", "rigatoni", "tagliatelle"]):
        tags.add("pasta")
    if any(k in txt for k in ["fish", "salmon", "tuna", "anchovy", "shrimp", "prawn", "octopus", "mussel"]):
        tags.add("seafood")
    if any(k in txt for k in ["beef", "pork", "chicken", "lamb", "sausage", "guanciale", "bacon"]):
        tags.add("meat")

    # Very bold veggie tag adding
    if "meat" not in tags and "seafood" not in tags:
        tags.add("vegetarian_candidate")

    return sorted(tags)

def build_document_schema(raw: str, sections: List[str], url: Optional[str] = None) -> Dict[str, Any]:
    sections = split_sections(raw, sections)

    dish_name = sections.get("name", "").strip()
    ingredients = parse_ingredients_block(sections.get("ingredients", ""))
    info = parse_info_block(sections.get("info", ""))
    steps = split_instructions(sections.get("instructions", ""))

    doc_id = stable_id(dish_name + "\n" + raw, prefix="doc")

    # Normalize into schema
    doc: Dict[str, Any] = {
        "doc_id": doc_id,
        "url": url,
        "dish_name": dish_name or None,
        "region": None,     # Future extension
        "doc_type": "recipe",
        "sections": {
            "ingredients": [f'{x["name"]}: {x["amount"]}'.strip(": ").strip() for x in ingredients],
            "steps": steps,
            "notes": "\n".join(info.get("notes", [])).strip() or None,
            "history": None,
            "variations": None,
            "techniques": None,
        },
        "servings": info.get("servings"),
        "prep_time": info.get("prep_time"),
        "cook_time": info.get("cook_time"),
        "total_time": info.get("total_time"),
        "tags": infer_tags(dish_name, ingredients, sections.get("instructions", "")),
        "parsed_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }

    return doc

def make_chunks(doc: Dict[str, Any], max_chars: int = 1200) -> List[Dict[str, Any]]:
    """
    Create section-aware chunks suitable for hybrid retrieval.
    Uses a simple char budget (works fine but can switch to token budgets later).
    """
    chunks: List[Dict[str, Any]] = []

    def add_chunk(section: str, text: str, idx: int, extra: Optional[Dict[str, Any]] = None):
        c = {
            "chunk_id": f'{doc["doc_id"]}#{section}#{idx}',
            "doc_id": doc["doc_id"],
            "dish_name": doc.get("dish_name"),
            "region": doc.get("region"),
            "doc_type": doc.get("doc_type"),
            "source": doc.get("source"),
            "url": doc.get("url"),
            "section": section,
            "tags": doc.get("tags", []),
            "text": text.strip(),
        }
        if extra:
            c.update(extra)
        chunks.append(c)

    # Ingredients: usually one chunk
    ing = doc["sections"].get("ingredients") or []
    if ing:
        ing_text = "Ingredients:\n" + "\n".join(f"- {x}" for x in ing)
        add_chunk("ingredients", ing_text, 0)

    # Notes: one chunk
    notes = doc["sections"].get("notes")
    if notes:
        add_chunk("notes", "Notes:\n" + notes, 0)

    # Steps: chunk in groups
    steps = doc["sections"].get("steps") or []
    if steps:
        buf: List[str] = []
        idx = 0
        for i, step in enumerate(steps, start=1):
            candidate = "\n".join(buf + [f"{i}. {step}"]).strip()
            if buf and len(candidate) > max_chars:
                add_chunk("steps", "Instructions:\n" + "\n".join(buf), idx, extra={"step_start": i - len(buf), "step_end": i - 1})
                idx += 1
                buf = [f"{i}. {step}"]
            else:
                buf.append(f"{i}. {step}")

        if buf:
            start = int(re.match(r"^(\d+)\.", buf[0]).group(1)) if re.match(r"^(\d+)\.", buf[0]) else None
            end = int(re.match(r"^(\d+)\.", buf[-1]).group(1)) if re.match(r"^(\d+)\.", buf[-1]) else None
            add_chunk("steps", "Instructions:\n" + "\n".join(buf), idx, extra={"step_start": start, "step_end": end})

    # Times/servings as a compact factual chunk
    facts = []
    if doc.get("servings") is not None:
        facts.append(f"Servings: {doc['servings']}")
    if doc.get("prep_time") is not None:
        facts.append(f"Prep time (min): {doc['prep_time']}")
    if doc.get("cook_time") is not None:
        facts.append(f"Cook time (min): {doc['cook_time']}")
    if doc.get("total_time") is not None:
        facts.append(f"Total time (min): {doc['total_time']}")
    if facts:
        add_chunk("info", "Info:\n" + "\n".join(f"- {x}" for x in facts), 0)

    return chunks

def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def write_jsonl(path: Path, records: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        for rec in records:
            file.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [10]:
INPUT_DIR = "./data/"
OUT_DIR = "./schema/"
MAX_CHUNK_CHARS = 60
SECTION_ORDER = ["name", "ingredients", "info", "instructions"]

filepaths: List[Path] = []
docs = Path(INPUT_DIR)
filepaths.extend(sorted([filepath for filepath in docs.glob("**/*.txt") if filepath.is_file()]))

documents: List[Dict[str, Any]] = []
chunks: List[Dict[str, Any]] = []

for file in filepaths:

    raw = read_text_file(file)
    doc = build_document_schema(raw=raw, sections=SECTION_ORDER, url=None)
    # keep original file
    doc["source_file"] = str(file)
    documents.append(doc)
    chunks.extend(make_chunks(doc, max_chars=MAX_CHUNK_CHARS))

out_dir = Path(OUT_DIR)
write_jsonl(out_dir / "documents.jsonl", documents)
write_jsonl(out_dir / "chunks.jsonl", chunks)

print(f"Wrote {len(documents)} documents to {out_dir / 'documents.jsonl'}")
print(f"Wrote {len(chunks)} chunks to {out_dir / 'chunks.jsonl'}")

Wrote 6651 documents to schema/documents.jsonl
Wrote 87147 chunks to schema/chunks.jsonl


## RAG Model

In [11]:
from pathlib import Path

import numpy as np

from dataclasses import dataclass
import faiss
from sentence_transformers import SentenceTransformer
import pickle
import math
import torch

/opt/miniconda3/envs/faiss_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
@dataclass(frozen=True)
class Chunk:
    text: str
    metadata: Dict[str, Any]

def _coerce_metadata(obj: Dict[str, Any]) -> Dict[str, Any]:
    """
    Accepts either:
      (A) {"text": "...", "metadata": {...}}
      (B) flat: {"text": "...", "chunk_id": "...", "doc_id": "...", ...}
    Returns metadata dict.
    """
    if "metadata" in obj and isinstance(obj["metadata"], dict):
        md = dict(obj["metadata"])
        # keep any other top-level fields (except text/metadata) as well
        for k, v in obj.items():
            if k not in {"text", "metadata"} and k not in md:
                md[k] = v
        return md

    # flat case: everything except text is metadata
    md = {k: v for k, v in obj.items() if k != "text"}
    return md

def load_chunks(data_path: str, limit: Optional[int] = None) -> List[Chunk]:
    """
    Loads JSONL where each line is a chunk-like object.
    Works with:
      - flat chunk lines (your example)
      - lines with {"text": "...", "metadata": {...}}
    """
    path = Path(data_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix.lower() != ".jsonl":
        raise ValueError("This loader expects a .jsonl file.")

    out: List[Chunk] = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and len(out) >= int(limit):
                break

            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)
            if not isinstance(obj, dict):
                continue

            text = (obj.get("text") or "").strip()
            if not text:
                continue

            md = _coerce_metadata(obj)
            md.setdefault("row_index", i)

            # Handy defaults for your schema
            md.setdefault("doc_id", obj.get("doc_id"))
            md.setdefault("chunk_id", obj.get("chunk_id"))
            md.setdefault("dish_name", obj.get("dish_name"))
            md.setdefault("section", obj.get("section"))
            md.setdefault("doc_type", obj.get("doc_type"))
            md.setdefault("source", obj.get("source"))

            out.append(Chunk(text=text, metadata=md))

    if not out:
        raise ValueError("No valid chunks found (no non-empty 'text' fields).")

    return out

In [13]:

@dataclass
class SimilarityIndex:
    index: faiss.Index
    embeddings: np.ndarray | None
    model_name: str

def _encode_texts(
    model: SentenceTransformer,
    texts: Sequence[str],
    batch_size: int = 64,
    device: str = "cpu",
) -> np.ndarray:
    emb = model.encode(
        list(texts),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,  # cosine similarity via inner product
        show_progress_bar=True,
        device=device,
    )
    emb = emb.astype("float32", copy=False)
    return emb

def build_similarity_index(
    texts: Sequence[str],
    enc_model_name: str = "all-MiniLM-L6-v2",
    device: str = "cpu",
    batch_size: int = 64,
) -> Tuple[SentenceTransformer, SimilarityIndex]:
    """
    Builds a FAISS index for cosine similarity using normalized embeddings and IndexFlatIP.
    """
    model = SentenceTransformer(enc_model_name, device=device)
    embeddings = _encode_texts(model, texts, batch_size=batch_size, device=device)

    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)

    return model, SimilarityIndex(index=index, embeddings=embeddings, model_name=enc_model_name)

def search(
    model: SentenceTransformer,
    sim: SimilarityIndex,
    query: str,
    top_k: int = 5,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns (scores, indices) each of shape (top_k,).
    Scores are cosine similarities in [-1, 1] (usually [0,1] with these models).
    """
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, idx = sim.index.search(q, top_k)
    return scores[0], idx[0]

In [14]:

_WORD_RE = re.compile(r"[A-Za-z0-9]+")

def tokenize(text: str) -> List[str]:
    return [m.group(0).lower() for m in _WORD_RE.finditer(text or "")]

@dataclass
class BM25Index:
    k1: float
    b: float
    avgdl: float
    doc_len: List[int]
    df: Dict[str, int]                 # document frequency
    idf: Dict[str, float]              # idf per term
    tfs: List[Dict[str, int]]          # term freq per document

def build_bm25(
    texts: Sequence[str],
    k1: float = 1.5,
    b: float = 0.75,
) -> BM25Index:
    tfs: List[Dict[str, int]] = []
    df: Dict[str, int] = {}
    doc_len: List[int] = []

    for text in texts:
        toks = tokenize(text)
        doc_len.append(len(toks))
        tf: Dict[str, int] = {}
        for t in toks:
            tf[t] = tf.get(t, 0) + 1
        tfs.append(tf)

        # update df with unique terms
        for t in tf.keys():
            df[t] = df.get(t, 0) + 1

    N = len(texts)
    avgdl = (sum(doc_len) / N) if N else 0.0

    # BM25+ style IDF (common stable variant)
    idf: Dict[str, float] = {}
    for term, n_qi in df.items():
        # log( (N - df + 0.5) / (df + 0.5) + 1 )
        idf[term] = math.log((N - n_qi + 0.5) / (n_qi + 0.5) + 1.0)

    return BM25Index(k1=k1, b=b, avgdl=avgdl, doc_len=doc_len, df=df, idf=idf, tfs=tfs)

def bm25_search(
    index: BM25Index,
    query: str,
    top_k: int = 50,
) -> Tuple[List[float], List[int]]:
    q_terms = tokenize(query)
    if not q_terms:
        return [], []

    scores: List[float] = [0.0] * len(index.tfs)

    for term in q_terms:
        idf = index.idf.get(term)
        if idf is None:
            continue

        for doc_id, tf in enumerate(index.tfs):
            f = tf.get(term, 0)
            if f == 0:
                continue

            dl = index.doc_len[doc_id]
            denom = f + index.k1 * (1.0 - index.b + index.b * (dl / index.avgdl if index.avgdl else 0.0))
            score = idf * (f * (index.k1 + 1.0)) / (denom if denom else 1.0)
            scores[doc_id] += score

    # top-k by score
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    ranked = [(i, s) for i, s in ranked if s > 0.0][:top_k]

    idx = [i for i, _ in ranked]
    sc = [s for _, s in ranked]
    return sc, idx

In [15]:

STORE_DIR = Path("schemas")
INDEX_PATH = STORE_DIR / "chunks.faiss"
META_PATH  = STORE_DIR / "chunks_meta.pkl"
def save_store(
    *,
    index: faiss.Index,
    chunks: List[Chunk],
    enc_model_name: str,
    bm25: BM25Index,
) -> None:
    STORE_DIR.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(INDEX_PATH))

    payload = {
        "enc_model_name": enc_model_name,
        "chunks": chunks,
        "bm25": bm25,
    }
    with META_PATH.open("wb") as f:
        pickle.dump(payload, f)

def load_store(device: str = "cpu") -> Tuple[SentenceTransformer, faiss.Index, List[Chunk], str, BM25Index]:
    if not INDEX_PATH.exists() or not META_PATH.exists():
        raise FileNotFoundError(
            f"Missing saved store. Expected {INDEX_PATH} and {META_PATH}."
        )

    index = faiss.read_index(str(INDEX_PATH))

    with META_PATH.open("rb") as f:
        payload: Dict[str, Any] = pickle.load(f)

    enc_model_name = payload["enc_model_name"]
    chunks = payload["chunks"]
    bm25 = payload["bm25"]

    embed_model = SentenceTransformer(enc_model_name, device=device)
    return embed_model, index, chunks, enc_model_name, bm25

In [16]:

@dataclass
class HybridConfig:
    k_bm25: int = 100
    k_dense: int = 100
    rrf_k: int = 60          # RRF constant, typical 60
    top_k: int = 10          # final results

def rrf_fuse(
    bm25_ranks: Sequence[int],
    dense_ranks: Sequence[int],
    rrf_k: int = 60,
) -> List[int]:
    """
    Reciprocal Rank Fusion:
      score(d) = Σ 1 / (rrf_k + rank_method(d))
    rank is 1-based.
    """
    scores: Dict[int, float] = {}

    for r, doc_id in enumerate(bm25_ranks, start=1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (rrf_k + r)

    for r, doc_id in enumerate(dense_ranks, start=1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (rrf_k + r)

    return [doc_id for doc_id, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

def hybrid_search(
    *,
    embed_model,
    dense: SimilarityIndex,
    bm25: BM25Index,
    query: str,
    cfg: HybridConfig,
) -> List[int]:
    # BM25 candidates (ranked)
    _, bm25_idx = bm25_search(bm25, query=query, top_k=cfg.k_bm25)

    # Dense candidates (ranked)
    _, dense_idx = search(embed_model, dense, query=query, top_k=cfg.k_dense)
    dense_idx_list = [int(i) for i in dense_idx.tolist() if int(i) >= 0]

    fused = rrf_fuse(bm25_idx, dense_idx_list, rrf_k=cfg.rrf_k)
    return fused[: cfg.top_k]

In [ ]:
OUT_DIR = Path("schema")
CHUNKS_PATH = OUT_DIR / "chunks.jsonl"
INDEX_PATH = OUT_DIR / "chunks.faiss"
META_PATH  = OUT_DIR / "chunks_meta.pkl"
REBUILD_STORE = True   # <-- flip to True to force rebuild + save

device = "cuda:0" if torch.cuda.is_available() else "cpu"

if REBUILD_STORE:
    chunks = load_chunks(CHUNKS_PATH)
    texts = [c.text for c in chunks]

    # Dense index
    embed_model, dense_sim = build_similarity_index(
        texts=texts,
        enc_model_name="all-MiniLM-L6-v2",
        device=device,
        batch_size=16,  # safer on macOS
    )

    # BM25 index
    bm25 = build_bm25(texts)

    # Save both
    save_store(
        index=dense_sim.index,
        chunks=chunks,
        enc_model_name=dense_sim.model_name,
        bm25=bm25,
    )

    dense = dense_sim
else:
    embed_model, faiss_index, chunks, enc_name, bm25 = load_store(device=device)
    dense = SimilarityIndex(index=faiss_index, embeddings=None, model_name=enc_name)

cfg = HybridConfig(
    k_bm25=100,  # candidates from BM25
    k_dense=100, # candidates from dense
    rrf_k=60,
    top_k=10,    # final top-k returned
)

while True:
    q = input("\nQuery> ").strip()
    if not q:
        continue
    if q.lower() in {"quit", "exit"}:
        break

    top_idxs = hybrid_search(
        embed_model=embed_model,
        dense=dense,
        bm25=bm25,
        query=q,
        cfg=cfg,
    )

    print("\nTop results:")
    for rank, i in enumerate(top_idxs, start=1):
        c = chunks[i]
        md = c.metadata
        text_preview = c.text.replace("\n", " ")[:140]
        print(
            f"{rank:2d}. "
            f"doc_id={md.get('doc_id')} | "
            f"chunk_id={md.get('chunk_id')} | "
            f"section={md.get('section')} | "
            f"dish_name={md.get('dish_name')}"
        )
        print(f"    {text_preview}...\n")


Batches:   0%|          | 26/5447 [00:11<40:18,  2.24it/s]


KeyboardInterrupt: 